# Bronze Layer Data Ingestion - Retail Store Sales (Incremental Load)

## Purpose
This notebook ingests raw CSV sales data from Unity Catalog volumes into the **bronze layer** Delta table using **incremental load** strategy.

## Process Flow
1. **Setup**: Import schema types and Delta table functions
2. **Read**: Load CSV from `/Volumes/sales/default/raw/retail_store_sales.csv` with explicit schema
3. **Watermark Check**: Determine the last loaded transaction date from existing bronze table
4. **Filter**: Only process records newer than the watermark
5. **Standardize**: Rename columns from "Title Case" to "snake_case"
6. **Append**: Add new records to Delta table (no overwrite)

## Incremental Strategy
- **Watermark Column**: `transaction_date`
- **Load Mode**: Append only new records
- **Deduplication**: Using transaction_id to prevent duplicates

## Source Data
- File: `/Volumes/sales/default/raw/retail_store_sales.csv`

## Output
- Delta table: `sales.bronze.retail_store_sales` with standardized column names

In [0]:
# ============================================================================
# Setup: Import Libraries and Define Table Name
# ============================================================================

# Hardcoded table name
table = "retail_store_sales"
full_table_name = f"sales.bronze.{table}"

# Import PySpark schema types for explicit schema definition
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, DateType, BooleanType
from pyspark.sql.functions import col, max as spark_max
from delta.tables import DeltaTable

In [0]:
# ============================================================================
# Step 1: Define Schema and Read Raw CSV Data
# ============================================================================
# Read retail_store_sales.csv from Unity Catalog volume
# Apply explicit schema to ensure data quality and type safety
# ============================================================================

schema = StructType([
    StructField("Transaction ID", StringType(), True),
    StructField("Customer ID", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Item", StringType(), True),
    StructField("Price Per Unit", DoubleType(), True),
    StructField("Quantity", DoubleType(), True),  # Keep as Double to preserve source precision
    StructField("Total Spent", DoubleType(), True),
    StructField("Payment Method", StringType(), True),
    StructField("Location", StringType(), True),
    StructField("Transaction Date", DateType(), True),
    StructField("Discount Applied", BooleanType(), True)
])

sales_df = spark.read \
                .option("header", True) \
                .schema(schema) \
                .csv(f"/Volumes/sales/default/raw/{table}.csv")

display(sales_df)

In [0]:
# ============================================================================
# Step 2: Incremental Load - Get Watermark and Filter New Records
# ============================================================================
# Check if bronze table exists and get the maximum transaction_date
# Only load records newer than the watermark
# ============================================================================

# Check if the bronze table exists
if spark.catalog.tableExists(full_table_name):
    # Get the maximum transaction_date from existing bronze table
    max_date_result = spark.table(full_table_name) \
        .select(spark_max("transaction_date").alias("max_date")) \
        .collect()[0]
    
    watermark = max_date_result["max_date"]
    
    if watermark:
        print(f"Watermark found: {watermark}")
        print(f"Loading only records after {watermark}")
        
        # Filter for records newer than the watermark
        sales_df_new = sales_df.filter(col("Transaction Date") > watermark)
        print(f"New records to load: {sales_df_new.count()}")
    else:
        print("Watermark is NULL, loading all records")
        sales_df_new = sales_df
else:
    print(f"Table {full_table_name} does not exist yet. Loading all records (initial load).")
    sales_df_new = sales_df

if sales_df_new.count() == 0:
    dbutils.notebook.exit("No new records to load")

display(sales_df_new)

In [0]:
# ============================================================================
# Step 3: Standardize Column Names
# ============================================================================
# Convert column names from "Title Case" to "snake_case" convention
# ============================================================================

sales_df_new = sales_df_new.withColumnsRenamed({
    "Transaction ID": "transaction_id",
    "Customer ID": "customer_id",
    "Category": "category",
    "Item": "item_id",
    "Price Per Unit": "price_per_unit",
    "Quantity": "quantity",
    "Total Spent": "total_spent",
    "Payment Method": "payment_method",
    "Location": "location",
    "Transaction Date": "transaction_date",
    "Discount Applied": "discount_applied"
})

display(sales_df_new)

In [0]:
# ============================================================================
# Step 4: Append to Bronze Layer (Incremental)
# ============================================================================
# Use merge operation to prevent duplicates based on transaction_id
# ============================================================================

if spark.catalog.tableExists(full_table_name):
    # Use merge to upsert and avoid duplicates
    bronze_table = DeltaTable.forName(spark, full_table_name)
    
    bronze_table.alias("target").merge(
        sales_df_new.alias("source"),
        "target.transaction_id = source.transaction_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
    
    print(f"Merged {sales_df_new.count()} records into {full_table_name}")
else:
    # Initial load
    sales_df_new.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(full_table_name)
    
    print(f"Created table {full_table_name} with {sales_df_new.count()} records")